# Problem Set 4: Disease Spread and the SIR Model

**Economics of Health, Human Capital, and Happiness (SOC POL 333)**
Northwestern University, Fall 2026 — due Wednesday 10/21 (beginning of class)

---

**Name:** *[REPLACE WITH YOUR NAME — double-click this cell, edit, then press Shift+Enter]*

---

Build your own epidemiological model of infectious disease spread — with vital dynamics, endemic cycles, and vaccination.

**Before you do anything else:** go to **File → Save a copy in Drive**.

## Part 0: Set-up

In this problem set you will build your own epidemiological model of infectious disease spread — the **SIR model** — entirely from difference equations. Unlike other problem sets there is no dataset to import: you will *generate* the data yourself by simulating a system of equations forward in time, one day at a time. (Many economic models — of growth, or asset pricing — are simulated exactly the same way.)

The SIR model divides a population of size $N$ into **S**usceptible, **I**nfectious, and **R**ecovered/immune, with $S_t + I_t + R_t = N$ in every period. Two parameters govern the disease:

- $\beta$ — the **transmission rate**: roughly, the rate at which an infectious person generates new infections among the susceptibles they contact.
- $\gamma$ — the **recovery rate**: the inverse of the average number of days someone stays infectious. If people are infectious for 8 days, $\gamma = 1/8$.

The key summary statistic is the **basic reproduction number** $R_0 = \beta/\gamma$: the average number of new infections caused by one infectious person in a fully susceptible population. If $R_0 > 1$ an epidemic grows; if $R_0 < 1$ it dies out.

We extend the textbook model in one important way: we add **vital dynamics** (births and deaths), so the model can reproduce not just a single pandemic wave but recurring **endemic** cycles, driven by new susceptible babies entering the population — much like measles before vaccination.

## The model

Let $\mu$ be the daily background birth/death rate (births and deaths balance, so population is constant), and $v$ the **vaccination rate**: the share of newborns vaccinated at birth, who enter directly as immune. The daily model is

$$S_{t+1} = S_t + \mu N (1-v) - \beta \frac{S_t I_t}{N} - \mu S_t$$
$$I_{t+1} = I_t + \beta \frac{S_t I_t}{N} - \gamma I_t - \mu I_t$$
$$R_{t+1} = R_t + \gamma I_t - \mu R_t + \mu N v$$

Each day: unvaccinated births add to $S$; some susceptibles get infected (the $\beta S_t I_t/N$ term — this is what makes the model nonlinear); some infectious people recover; vaccinated newborns go straight to $R$; and everyone faces the small background death rate $\mu$.

Illustrative parameters, assuming 75-year life expectancy so $\mu = 1/(75 \times 365)$:

| Disease | $R_0$ | Infectious period (days) | Implied $\beta = R_0 \gamma$ |
|---|---|---|---|
| Seasonal influenza | 1.3 | 5 | 0.26 |
| COVID-19 (original) | 3 | 10 | 0.30 |
| COVID-19 (Delta) | 6 | 10 | 0.60 |
| Measles | 15 | 8 | 1.875 |

These are approximate. The goal is to understand how the model behaves as $R_0$ changes, not to pin down a "true" number.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

print("Ready!")

## Part 1: Building the simulation

The function below is the whole model. Read it line by line and check each line against the three equations above — the loop *is* the difference-equation system.

In [ ]:
def sir(beta, gamma, v=0.0, N=1_000_000, I0=1, years=40, mu=1/(75*365)):
    """Simulate the SIR model with vital dynamics, day by day.

    beta  transmission rate      gamma recovery rate (1/infectious days)
    v     share of newborns vaccinated (enter directly as immune)
    N     population size        I0    initially infectious people
    """
    days = int(years * 365)
    S, I, R = N - I0, I0, 0
    rows = []
    for day in range(days + 1):
        rows.append({"day": day, "S": S, "I": I, "R": R})
        new_infections = min(beta * S * I / N, S)   # can't infect more people than are susceptible
        S_next = S + mu * N * (1 - v) - new_infections - mu * S
        I_next = I + new_infections - gamma * I - mu * I
        R_next = R + gamma * I - mu * R + mu * N * v
        S, I, R = S_next, I_next, R_next
    out = pd.DataFrame(rows)
    out["year"] = out.day / 365
    return out

# Measles parameters, no vaccination, one infectious person in a fully susceptible population
measles = sir(beta=1.875, gamma=1/8, v=0)

# Check: does the population add up in every period?
measles["check"] = measles.S + measles.I + measles.R
print(measles.check.describe()[["min", "mean", "max"]])

**Question 1.** Confirm from the output above that $S_t + I_t + R_t = N$ (up to rounding) in every period. Why is this a useful check to run before looking at any results?

**Answer:** *[your answer here]*

In [ ]:
plt.figure(figsize=(9, 5))
for col, label in [("S", "Susceptible"), ("I", "Infectious"), ("R", "Recovered/immune")]:
    plt.plot(measles.year, measles[col], label=label)
plt.xlabel("Year"); plt.ylabel("Number of people")
plt.title("SIR model with vital dynamics — measles parameters, no vaccination")
plt.legend()
plt.show()

# The initial wave dwarfs the endemic cycles, so also look at I alone, and at years 10-40
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(measles.year, measles.I)
axes[0].set_title("Infectious over time"); axes[0].set_xlabel("Year")
late = measles[measles.year >= 10]
axes[1].plot(late.year, late.I)
axes[1].set_title("Endemic phase (years 10-40)"); axes[1].set_xlabel("Year")
plt.show()

**Question 2. Describe what you see.** You should see a large initial epidemic wave (the "pandemic"), followed by smaller recurring waves (the "endemic" phase) as births replenish the pool of susceptibles. Describe the pattern in your own words, and explain *why* births are what generate the recurring waves.

**Answer:** *[your answer here]*

## Part 2: Comparing diseases — the role of $R_0$

**Question 3.** Run the simulation (no vaccination) for all four diseases in the table and plot their infectious paths together.

In [ ]:
diseases = {
    "Seasonal influenza":   dict(beta=0.26,  gamma=1/5),
    "COVID-19 (original)":  dict(beta=0.30,  gamma=1/10),
    "COVID-19 (Delta)":     dict(beta=0.60,  gamma=1/10),
    "Measles":              dict(beta=1.875, gamma=1/8),
}

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
summary = []
for ax, (name, pars) in zip(axes.flat, diseases.items()):
    epi = sir(**pars, v=0)
    ax.plot(epi.year, epi.I)
    ax.set_title(f"{name}  ($R_0$ = {pars['beta']/pars['gamma']:.1f})")
    ax.set_xlabel("Year"); ax.set_ylabel("Infectious")
    summary.append({"disease": name,
                    "R0": pars["beta"] / pars["gamma"],
                    "peak infectious": round(epi.I.max()),
                    "peak year": round(epi.loc[epi.I.idxmax(), "year"], 2),
                    "immune at year 40 (%)": round(100 * epi.R.iloc[-1] / 1_000_000, 1)})
plt.tight_layout(); plt.show()

pd.DataFrame(summary)

**Question 4. Compare the four diseases.** How does $R_0$ affect (a) how quickly the initial wave grows, (b) the size of the initial peak, (c) how much of the population is ultimately infected, and (d) whether — and how strongly — the disease settles into recurring endemic cycles versus dying out?

Relate your answer to the mechanics of the model: **which term in the difference equations is doing the work here?**

**Answer:** *[your answer here]*

**Question 5. Final epidemic size.** There is a well-known formula relating $R_0$ to the fraction of the population eventually infected in a single outbreak (with no vital dynamics), implicitly defined by

$$z = 1 - e^{-R_0 z}$$

You do not need to derive it, but solve it numerically for each of your four $R_0$ values and compare to the simulated cumulative infections before the endemic phase begins. Do they roughly match?

In [ ]:
def final_size(R0, tol=1e-10):
    """Solve z = 1 - exp(-R0 * z) by repeated substitution."""
    if R0 <= 1:
        return 0.0
    z = 0.5
    for _ in range(10_000):
        z_new = 1 - np.exp(-R0 * z)
        if abs(z_new - z) < tol:
            break
        z = z_new
    return z

rows = []
for name, pars in diseases.items():
    R0 = pars["beta"] / pars["gamma"]
    epi = sir(**pars, v=0)
    # cumulative infections just after the initial wave, before births matter much
    first_trough = epi.loc[epi.year.between(0, 10), "I"].idxmin()
    rows.append({"disease": name, "R0": round(R0, 2),
                 "predicted z (formula)": round(final_size(R0), 3),
                 "simulated share infected in wave 1": round(epi.loc[first_trough, "R"] / 1_000_000, 3)})
pd.DataFrame(rows)

**Question 5 (continued).** Do the formula and the simulation roughly agree? Where they differ, why might that be (think about what the formula assumes that our model does not)?

**Answer:** *[your answer here]*

## Part 3: The effect of vaccination

**Question 6.** The **herd immunity threshold** is the vaccination rate above which the disease cannot sustain an outbreak: $v_c = 1 - 1/R_0$. Compute it for measles, then re-run the measles simulation for $v = 0$, a rate below $v_c$, a rate just above $v_c$, and a high rate ($v = 0.99$). We simulate 100 years here, because with vaccination the endemic waves can take decades to return.

In [ ]:
R0_measles = 1.875 / (1/8)
v_c = 1 - 1 / R0_measles
print(f"Measles R0 = {R0_measles:.1f}, herd immunity threshold v_c = {v_c:.3f} ({v_c:.1%})")

plt.figure(figsize=(9, 5))
for v in [0.0, 0.80, 0.95, 0.99]:
    epi = sir(beta=1.875, gamma=1/8, v=v, years=100)
    plt.plot(epi.year, epi.I, label=f"v = {v:.0%}" + ("  (below $v_c$)" if v < v_c else "  (above $v_c$)"))
plt.xlabel("Year"); plt.ylabel("Infectious"); plt.yscale("symlog")
plt.title("Measles under different vaccination rates, 100 years (log scale)")
plt.legend(); plt.show()

**Question 7. Interpret your results.** What happens to the size of the initial pandemic wave as $v$ increases? What happens to the endemic cycles once $v$ crosses $v_c$? Does the disease ever fully disappear from your simulation, or does it merely become rare — and why?

**Answer:** *[your answer here]*

**Question 8.** Real-world MMR vaccination coverage needed to reliably prevent outbreaks is often cited as roughly **95%**. Compare that to the $v_c$ you calculated. Does the model's threshold match the commonly cited figure? If not, what modelling assumptions might explain the discrepancy (vaccine efficacy below 100%, non-random mixing, waning immunity, imported cases)?

**Answer:** *[your answer here]*

**Question 9.** Repeat the vaccination comparison for seasonal influenza, using its own $R_0$ and $v_c$. **Why is a much lower vaccination rate sufficient to control flu than measles?** Connect your answer to the formula for $v_c$.

In [ ]:
# Your flu vaccination comparison here


**Answer:** *[your answer here]*

## Suggested additional questions (optional)

1. **Imperfect vaccines.** Modify the model so vaccination only *reduces* susceptibility — vaccinated individuals face a force of infection multiplied by $(1-e)$, with efficacy $e = 0.9$. Add a fourth compartment $V$, or adjust the force-of-infection term. How does imperfect efficacy change the vaccination rate needed for herd immunity?
2. **A more realistic introduction shock.** Start with 50 infectious travellers arriving at once instead of one case. Does the qualitative pattern change, or just the timing?
3. **Waning immunity (SIRS).** Let a small fraction $\omega$ of the recovered return to $S$ each period. How does this change the long-run behaviour?
4. **Matching real outbreaks (harder).** Pick a documented outbreak (measles in under-vaccinated US communities in 2014–15 or 2019, or early COVID-19 somewhere specific), find reported case counts, and try to choose $\beta$, $\gamma$ and initial conditions that roughly match early growth. Why is an exact match difficult?
5. **Policy discussion.** Given your simulations, why do public health officials worry about vaccination rates dropping even modestly below the herd immunity threshold, rather than treating "high" coverage as good enough? What does the model imply about a slow decline in MMR coverage within a subpopulation, even if the national average stays high?

## Submitting

Two steps, in this order:

1. **Submit your notebook** — type your name in the box in the cell below, then run it (▶). A Google popup will ask you to sign in: **choose your Northwestern account** (…@u.northwestern.edu). Submissions from non-Northwestern accounts are rejected. You should see "Submitted!".
2. **Submit your PDF on Canvas** — **File → Print → Save as PDF**, then upload the PDF to this week's Canvas assignment. The Canvas submission is the official one used for grading and deadlines.

(Before submitting, make sure your name is filled in at the top of the notebook and that you have run everything top to bottom via **Runtime → Run all** with no errors.)

In [ ]:
#@title ▶ Submit this notebook { display-mode: "form" }
#@markdown Type your full name, then run this cell. Sign in with your **Northwestern** Google account when prompted.
STUDENT_NAME = "" #@param {type:"string"}

SUBMIT_URL = "https://script.google.com/macros/s/AKfycbzRoLWLJHSItqAVf8GHEmKuvarpoLybC9bAsgnud5hoFDG4W-9YsrSHZl_Oo3cMnoFP/exec"
PS_ID = "PS04"

import json, subprocess, requests
from google.colab import auth

name = STUDENT_NAME.strip()
if not name or "REPLACE" in name.upper():
    raise SystemExit("⚠️ Please type your name in the box above, then run this cell again.")

# Sign in with Google (popup) and collect proof of identity for the server to verify
auth.authenticate_user()
id_token, access_token = None, None
try:
    import google.auth
    from google.auth.transport.requests import Request
    creds, _ = google.auth.default()
    creds.refresh(Request())
    id_token = getattr(creds, "id_token", None)
    access_token = getattr(creds, "token", None)
except Exception:
    pass
if not id_token:
    try:
        id_token = subprocess.run(["gcloud", "auth", "print-identity-token"],
                                  capture_output=True, text=True, timeout=30).stdout.strip() or None
    except Exception:
        pass
if not (id_token or access_token):
    raise SystemExit("⚠️ Could not verify your Google sign-in. Run the cell again; if it persists, email the TA.")

from google.colab import _message
notebook = _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]

resp = requests.post(
    SUBMIT_URL,
    json={"token": "e3h-2026-nu", "name": name, "ps": PS_ID,
          "id_token": id_token or "", "access_token": access_token or "",
          "notebook": notebook},
    timeout=120,
)
result = resp.json()
if result.get("ok"):
    print(f"✅ Submitted as {result['email']} — received for {result['ps']}.")
    print("Now also upload your PDF to Canvas (File → Print → Save as PDF).")
else:
    print("❌ Submission failed:", result.get("error"))
    print("Try again in a minute; if it keeps failing, email the TA and submit on Canvas only.")